In [1]:
import numpy as np
import pandas as pd

from scformer.utils import *
from scformer.model import *
from warnings import filterwarnings
import random
import os
import torch
import torch.cuda as cuda
from scipy import sparse

In [2]:
filterwarnings("ignore")
seed = 0
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

In [3]:
gene_cell = sparse.load_npz('data/example/RNA.npz')
gene_names = pd.DataFrame(np.load('data/example/gene_name.npy', allow_pickle=True))
true_label = np.load('data/example/label500.npy', allow_pickle=True)

gene_cell.obs_names = gene_names[0]

RNA_matrix = gene_cell

cell_num = RNA_matrix.shape[1]
gene_num = RNA_matrix.shape[0]

In [4]:
initial_pre = initial_clustering(RNA_matrix)

	When the number of cells is less than or equal to 500, it is recommended to set the resolution value to 0.2.
	When the number of cells is within the range of 500 to 5000, the resolution value should be set to 0.5.
	When the number of cells is greater than 5000, the resolution value should be set to 0.8.
         Falling back to preprocessing with `sc.pp.pca` and default params.


In [5]:
cluster_ini_num = len(set(initial_pre))
ini_p1 = [int(i) for i in initial_pre]
# partite the data into batches
indices, Node_Ids, dic = batch_select_whole(RNA_matrix)
n_batch = len(indices)

Partitioning the data into batches. Please wait...


Processing Batches: 100%|██████████| 17/17 [00:00<00:00, 21.75it/s]


In [6]:
device = torch.device("cuda" if cuda.is_available() else "cpu")
# 训练 NodeDimensionReduction 模型
node_model = NodeDimensionReduction(
    RNA_matrix=RNA_matrix,
    indices=indices,
    n_hid=104,
    n_heads=8,
    n_layers=3,
    labsm=0.1,
    lr=0.0005,
    wd=0.1,
    device=device,
    num_types=2,
    num_relations=2,
    epochs=100,
    num_clusters=cluster_ini_num
)
gnn, cell_emb, gene_emb, h = node_model.train_model(n_batch=n_batch)

The training process for the NodeDimensionReduction model has started. Please wait.


100%|██████████| 100/100 [01:23<00:00,  1.20it/s]

The training for the NodeDimensionReduction model has been completed.


In [7]:
# 保存 NodeDimensionReduction 训练好的模型和聚类中心
node_model.save_model('node_dimension_reduction_model.pth')

In [8]:
# 加载 NodeDimensionReduction 模型和聚类中心
# 假设您知道 n_hid 和 num_clusters
loaded_node_model = NodeDimensionReduction.load_model(
    file_path='node_dimension_reduction_model.pth',
    device=device,
    RNA_matrix=RNA_matrix,
    indices=indices
)

In [9]:
scformer_model = ScFormer(
    gnn=loaded_node_model.gnn,
    cluster_centers=loaded_node_model.cluster_centers,
    labsm=0.1,
    n_hid=10,
    device=device
)

# 预测并保存结果
ScFormer_result = scformer_model.predict(
    RNA_matrix=RNA_matrix,
    indices=indices,
    nodes_id=Node_Ids,  # 您的节点 ID 列表
    cell_size=30
)

Prediction Batches: 100%|██████████| 17/17 [00:00<00:00, 28.57it/s]


In [10]:
ScFormer_result

{'pred_label': array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1

In [7]:
ScFormer_model = ScFormer(gnn=gnn, h=h, labsm=0.1, n_hid=104, n_batch=n_batch, device=device, lr=0.0005, wd=0.1,
                          num_epochs=50)
ScFormer_gnn, _, _, _ = ScFormer_model.train_model(indices=indices, RNA_matrix=RNA_matrix, ini_p1=ini_p1)

The training process for the scformer model has started. Please wait.
Forward pass started.


Epochs: 100%|██████████| 50/50 [00:40<00:00,  1.22it/s]

The training for the scformer model has been completed.
The training for the scformer model has been completed.


In [11]:
# from scformer.model import ScFormer_pred

# ScFormer_result = ScFormer_pred(RNA_matrix, gnn=ScFormer_gnn, indices=indices,
#                                 nodes_id=Node_Ids, device=device)
# Save numpy arrays to files
output_file = 'data/example/output'
np.save(output_file + "/Node_Ids.npy", Node_Ids)
np.save(output_file + "/pred.npy", ScFormer_result['pred_label'])
np.save(output_file + "/cell_embedding.npy", ScFormer_result['cell_embedding'])

In [12]:
pred_label = ScFormer_result['pred_label']
p_score, labels = purity_score(np.array(true_label), pred_label)
e = Entropy(np.array(pred_label, dtype='int64'), np.array(labels, dtype='int64'))
print("purity:%.4f" % p_score)
print("NMI:%.4f" % normalized_mutual_info_score(true_label, pred_label))
print("Entropy:%.4f" % e)

purity:0.9800
NMI:0.0000
Entropy:0.0980


In [13]:
Node_Ids = np.load("data/example/output/Node_Ids.npy")
pred = np.load('data/example/output/pred.npy')
pred_ = pd.DataFrame(pred, index=Node_Ids)
pred_sorted = pred_.sort_index()
pred_sorted

,0
0,1
1,1
2,1
3,1
4,1
...,...
495,1
496,1
497,1
498,1


In [14]:
true_label = np.load('data/example/label500.npy', allow_pickle=True)
true_label

array(['CD4+ T naive', 'CD4+ T naive', 'CD4+ T naive', 'CD4+ T naive',
       'CD4+ T naive', 'CD4+ T naive', 'CD4+ T naive', 'CD4+ T naive',
       'CD4+ T naive', 'CD4+ T naive', 'CD4+ T naive', 'CD4+ T naive',
       'CD4+ T naive', 'CD4+ T naive', 'CD4+ T naive', 'CD4+ T naive',
       'CD4+ T naive', 'CD4+ T naive', 'CD4+ T naive', 'CD4+ T naive',
       'CD4+ T naive', 'CD4+ T naive', 'CD4+ T naive', 'CD4+ T naive',
       'CD4+ T naive', 'CD4+ T naive', 'CD4+ T naive', 'CD4+ T naive',
       'CD4+ T naive', 'CD4+ T naive', 'CD4+ T naive', 'CD4+ T naive',
       'CD4+ T naive', 'CD4+ T naive', 'CD4+ T naive', 'CD4+ T naive',
       'CD4+ T naive', 'CD4+ T naive', 'CD4+ T naive', 'CD4+ T naive',
       'CD4+ T naive', 'CD4+ T naive', 'CD4+ T naive', 'CD4+ T naive',
       'CD4+ T naive', 'CD4+ T naive', 'CD4+ T naive', 'CD4+ T naive',
       'CD4+ T naive', 'CD4+ T naive', 'CD4+ T naive', 'CD4+ T naive',
       'CD4+ T naive', 'CD4+ T naive', 'CD4+ T naive', 'CD4+ T naive',
      

In [15]:
pre_index = np.array(pred_sorted)
pre_index.reshape(500, )

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [16]:
normalized_mutual_info_score(true_label, pre_index.reshape(500, ))

0.0